In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

In [2]:
# --- 1) Cargar dataset ---
datos = pd.read_csv("caracteristicas.csv")

In [3]:
# --- 2) Definir X (características) e y (etiqueta) ---
X = datos.drop(columns=["hipoxia", "marca_tiempo", "estacion"], errors="ignore")
y = datos["hipoxia"]

In [4]:
# --- 3) División entrenamiento / prueba (estratificada) ---
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
# A) GRIDSEARCH con UN SOLO SPLIT 
# ===========================
indices = np.arange(len(y_entrenamiento))
ind_entrenamiento, ind_validacion = train_test_split(
    indices, test_size=0.33, stratify=y_entrenamiento, random_state=42
)
cv = zip([ind_entrenamiento], [ind_validacion])
parametros = {
    'criterion': ('gini', 'entropy', 'log_loss'),
    'max_features': ('sqrt', 'log2', None),
    'n_estimators': np.arange(10, 101, 10)
}
bosque = RandomForestClassifier(random_state=42, n_jobs=-1)
busqueda = GridSearchCV(bosque, parametros, cv=cv, n_jobs=-1)
busqueda.fit(X_entrenamiento, y_entrenamiento)
print("=== Resultados Grid (CV con 1 partición) ===")
print("Mejores hiperparámetros:", busqueda.best_params_)
print("Exactitud media en validación (mejor combinación):", busqueda.best_score_)

# Tabla de resultados (opcional)
resultados_1split = pd.DataFrame(busqueda.cv_results_)
print("\nVista rápida de resultados (primeras filas):")
print(resultados_1split.head(10))
resultados_1split.to_csv("resultados_rf_cv1split.csv", index=False)
# Entrenar modelo final con esos hiperparámetros y evaluar en prueba
modelo_final_1split = RandomForestClassifier(
    criterion=busqueda.best_params_['criterion'],
    max_features=busqueda.best_params_['max_features'],
    n_estimators=int(busqueda.best_params_['n_estimators']),
    random_state=42,
    n_jobs=-1
)
modelo_final_1split.fit(X_entrenamiento, y_entrenamiento)
print("\nExactitud en prueba (modelo CV 1 partición):", modelo_final_1split.score(X_prueba, y_prueba))

# ===========================
# B) GRIDSEARCH con CV=10 (más robusto, también como mostró el profesor)
# ===========================
bosque_cv = RandomForestClassifier(random_state=42, n_jobs=-1)
busqueda_cv = GridSearchCV(bosque_cv, parametros, cv=10, n_jobs=-1)
busqueda_cv.fit(X_entrenamiento, y_entrenamiento)

print("\n=== Resultados Grid (CV=10) ===")
print("Mejores hiperparámetros:", busqueda_cv.best_params_)
print("Exactitud media CV=10 (mejor combinación):", busqueda_cv.best_score_)

# Tabla completa de resultados CV=10
resultados_cv10 = pd.DataFrame(busqueda_cv.cv_results_)
print("\nVista rápida TOP-10 combinaciones por media en validación:")
print(resultados_cv10.sort_values("mean_test_score", ascending=False)
                     .head(10)[["mean_test_score","std_test_score",
                                "param_criterion","param_max_features","param_n_estimators"]])
resultados_cv10.to_csv("resultados_rf_cv10.csv", index=False)

# Entrenar modelo final con los mejores hiperparámetros de CV=10 y evaluar en prueba
modelo_final_cv10 = RandomForestClassifier(
    criterion=busqueda_cv.best_params_['criterion'],
    max_features=busqueda_cv.best_params_['max_features'],
    n_estimators=int(busqueda_cv.best_params_['n_estimators']),
    random_state=42,
    n_jobs=-1
)
modelo_final_cv10.fit(X_entrenamiento, y_entrenamiento)
print("\nExactitud en prueba (modelo CV=10):", modelo_final_cv10.score(X_prueba, y_prueba))


=== Resultados Grid (CV con 1 partición) ===
Mejores hiperparámetros: {'criterion': 'gini', 'max_features': None, 'n_estimators': np.int64(90)}
Exactitud media en validación (mejor combinación): 0.9980400108137334

Vista rápida de resultados (primeras filas):
   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       1.097227           0.0         0.197099             0.0   
1       2.074228           0.0         0.186871             0.0   
2       2.825899           0.0         0.206493             0.0   
3       3.498670           0.0         0.172359             0.0   
4       4.401966           0.0         0.175827             0.0   
5       5.514182           0.0         0.163992             0.0   
6       6.187227           0.0         0.295948             0.0   
7       7.008551           0.0         0.234497             0.0   
8       8.077566           0.0         0.183415             0.0   
9       9.482004           0.0         0.197348             0.0   

  